# DDIM baseline + covariate CFG (TCGA, L1000 landmark genes)

Branch `cfg-demographics`. Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.

* **baseline** – tissue-conditioned DDIM (Lacan et al. replication, original label scheme)
* **cfg** – cancer type + sex + race + continuous age, with classifier-free guidance

**Storage.** Training writes only to Colab's local disk (`/content/runs`). Every ~20 min the latest
checkpoint is uploaded to Google Drive through the Drive API (`My Drive/genaiproject_backups_standard/<model>/`);
an upload only counts once Drive has stored it. Only the latest copy is kept (~1 GB per model).
Samples, metrics and logs are uploaded when a model finishes.

**If Colab disconnects:** reconnect and run cells 1–4 again. Each model resumes from its local
checkpoint, or on a fresh machine from the Drive backup (losing at most ~20 min of training).

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Sign in so the training job can upload backups to Google Drive (Drive API, not a mounted folder)
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
build('drive', 'v3', cache_discovery=False).files().list(pageSize=1, fields='files(id)').execute()
print('Drive API access OK')
EXP_ROOT = '/content/runs'          # local disk: fast, and never blocks on Drive
!mkdir -p {EXP_ROOT}

In [ ]:
# 3. Get the code (clone once, pull on later sessions) + the one missing dependency
import os
if not os.path.exists('/content/genaiproject'):
    !git clone -q -b cfg-demographics https://github.com/mzsanborn/genaiproject.git /content/genaiproject
else:
    !git -C /content/genaiproject pull -q
%cd /content/genaiproject
!git log --oneline -1
!pip install -q pytorch_warmup

## 4. Launch training

Runs in the background so the notebook stays usable; output is appended to `/content/runs/<model>.log`.
`WHICH = 'both'` trains the baseline, then the CFG model.

Each model **resumes automatically** (local checkpoint, else Drive backup, else fresh start).
Only run this cell when nothing is training (cell 5 shows that).
⚠️ `RESTART = True` **deletes** the local runs and starts over (the Drive backup is replaced at the next upload).

In [ ]:
WHICH = 'baseline'     # 'baseline', 'cfg' or 'both'
RESTART = False    # True = delete existing runs and train from scratch

models = ['baseline', 'cfg'] if WHICH == 'both' else [WHICH]
env = f"EXP_ROOT={EXP_ROOT} RESUME={0 if RESTART else 1}"
cmd = ' && '.join(f"{env} bash run_colab.sh {m} >> {EXP_ROOT}/{m}.log 2>&1" for m in models)
!nohup bash -c "{cmd}" > /dev/null 2>&1 &
print('launched:', models)

In [ ]:
# 5. Progress (re-run this cell whenever)
import os, re, subprocess

def tail(path, n_bytes=200_000):
    with open(path, 'rb') as f:
        f.seek(0, os.SEEK_END)
        f.seek(max(0, f.tell() - n_bytes))
        return f.read().decode(errors='ignore').splitlines()

for m in ['baseline', 'cfg']:
    log = f'{EXP_ROOT}/{m}.log'
    if not os.path.exists(log):
        print(f'{m}: not started')
        continue
    lines = tail(log)
    step = [l for l in lines if 'epoch:' in l and 'step:' in l]
    starts = [l for l in lines if l.startswith(('Resuming', 'Starting'))]
    errs = [l for l in lines if 'Traceback' in l or 'Error' in l]
    print(f'--- {m}')
    if starts:
        print('   ' + starts[-1])
    if step:
        print('   ' + re.sub(r'^.*? - (epoch:)', r'\1', step[-1]))
    backups = [l for l in lines if '[backup]' in l and not l.startswith('INFO:root')]
    if backups:
        print('   last backup: ' + re.sub(r'^.*? - \[backup\]', '[backup]', backups[-1])[:200])
    print('   last line:', lines[-1][:200] if lines else '')
    if errs:
        print('   recent errors (may be from an earlier, already-resumed run):')
        for e in errs[-3:]:
            print('     ' + e[:200])

print()
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 6. Results once a model has finished (5 sampling runs: mean, std)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, m in enumerate(['baseline', 'cfg']):
    d = f'{EXP_ROOT}/{m}/logs/tcga'
    try:
        res = np.load(f'{d}/dict_res_metrics_5runs.npy', allow_pickle=True).item()
        print(m, {k: [round(float(x), 4) for x in v] for k, v in res.items()})
    except FileNotFoundError:
        print(m, ': no final metrics yet')
    try:
        loss = pd.read_csv(f'{d}/loss.csv').drop_duplicates('epoch', keep='last')
        ax[i].plot(loss['epoch'], loss['loss']); ax[i].set_title(f'{m} training loss'); ax[i].set_xlabel('epoch')
    except FileNotFoundError:
        pass
plt.tight_layout()

## 7. Reverse validation (train on synthetic, test on real)

Run after a model has **finished** (its samples are in Drive). One multi-task MLP (cancer type, sex,
race, age) is trained on real data (reference) and on the model's synthetic data, and tested on the
real test patients; 5 seeds. Results are printed and uploaded next to the model's backup as
`reverse_validation.csv`. Add `--smoke` first for a ~1-minute check that everything runs.

In [ ]:
MODEL = 'baseline'   # 'baseline' or 'cfg'
%cd /content/genaiproject
!python reverse_validation.py --model {MODEL} --from_backup genaiproject_backups_standard --runs 5

Notes
* Fréchet distance here is computed in a PCA space fitted on real training data (the original authors'
  pretrained classifier weights are not in the repo), so it is not numerically comparable to the paper.
* Kill a run: `!pkill -f main_ddim.py`
* Backups: `My Drive/genaiproject_backups_standard/<model>/` (latest checkpoint, then results when finished)